# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I am building a CTR / engagement opportunity baseline. The rule is simple: pages with visible demand but weak CTR, weak position-adjusted CTR, or weak engagement are candidates for human review. I am not using product flags or future-window features. My score is based on observable data available at scoring time.

Reason codes I will use:
- `high_impressions_low_ctr` — high visibility but low CTR for its position
- `strong_position_weak_ctr` — excellent position but weak CTR
- `visible_low_engagement` — clicks occur but visitors do not engage
- `stale_visible_page` — still visible but old and decaying
- `position_opportunity_gap` — good engagement but low rank
- `monitor` — all other pages

The main logic behind this rule is position-adjusted CTR. FlyRank's `needs_ctr_fix` logic keys off CTR being weak relative to rank, so I am checking whether that pattern really appears in this dataset. If it does, it is a valid baseline signal.


In [ ]:
reason_codes = {
    'high_impressions_low_ctr': 'High visibility but low CTR for its position tier',
    'strong_position_weak_ctr': 'Very strong position but weak CTR',
    'visible_low_engagement': 'Users click but do not engage with the content',
    'stale_visible_page': 'Old content still visible but decaying',
    'position_opportunity_gap': 'Engaged page but ranks too deep',
    'monitor': 'Borderline or low-priority case'
}

for code, meaning in reason_codes.items():
    print(f'{code}: {meaning}')


## 2. Build the ranked queue (writes the CSV)

For each page, I calculate expected CTR by position tier, then compare each page's actual CTR to the median expected CTR for that tier. The gap tells me whether a page is underperforming relative to rank. I then combine the CTR gap with visibility and position strength into one weighted score. Pages with stronger visibility and stronger negative CTR gap get ranked higher.

The scoring logic is:
- calculate expected CTR by tier: top_3, page_1, striking, page_3_5
- compute `CTR_gap = actual_ctr - expected_ctr_for_position_tier`
- normalize CTR gap to 0–1
- normalize impressions to percentile rank
- normalize position strength (better positions get higher values)
- final score = (CTR_gap × 0.40) + (visibility_percentile × 0.40) + (position_strength × 0.20), scaled to 0–100

I keep only pages with enough data to trust the signal: impressions >= 500, content age >= 90 days, and average position between 1 and 49. I then assign one reason code and one suggested action per page, rank everything, and write the queue to `work/outputs/baseline_action_score.csv`.


In [ ]:
from pathlib import Path
import json

import pandas as pd

from scripts.ml_utils import normalize, percentile_rank

# Find the repo root in a robust way so this works in both local and Colab runs
candidates = [Path.cwd(), Path.cwd().parent]
root = None
for candidate in candidates:
    data_path = candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    if data_path.exists():
        root = candidate
        break
if root is None:
    raise FileNotFoundError('Could not find the repository root or the starter CSV file.')

raw_path = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
output_dir = root / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_path)
print(f'Started with {len(df):,} rows')

# Keep only rows with enough data and real visibility
df = df[(df['impressions_90d'] >= 500) & (df['content_age_days'] >= 90) & (df['avg_position'] > 0) & (df['avg_position'] < 50)].copy()
print(f'After filters: {len(df):,} rows')

# Position tier mapping
def position_tier(position):
    if position <= 3:
        return 'top_3'
    elif position <= 10:
        return 'page_1'
    elif position <= 20:
        return 'striking'
    else:
        return 'page_3_5'

df['position_tier'] = df['avg_position'].apply(position_tier)

# Median expected CTR by position tier
expected_ctr_by_tier = df.groupby('position_tier')['ctr'].median().to_dict()
df['expected_ctr'] = df['position_tier'].map(expected_ctr_by_tier)
df['ctr_gap'] = df['ctr'] - df['expected_ctr']

tier_summary = df.groupby('position_tier').agg(
    pages=('content_id', 'count'),
    median_ctr=('ctr', 'median')
).reset_index()
print('Position tier summary:')
print(tier_summary.to_string(index=False))

# Normalize signals to a common 0-1 scale
df['ctr_gap_normalized'] = normalize(df['ctr_gap'])
df['visibility_percentile'] = percentile_rank(df['impressions_90d'])
df['position_normalized'] = (df['avg_position'] - 1) / 49
df['position_strength'] = 1 - df['position_normalized']

# Final baseline score on a 0-100 scale
df['baseline_score'] = (
    (df['ctr_gap_normalized'] * 0.40)
    + (df['visibility_percentile'] * 0.40)
    + (df['position_strength'] * 0.20)
) * 100
df = df.sort_values('baseline_score', ascending=False).reset_index(drop=True)
df['baseline_rank'] = df.index + 1

print('Baseline score summary:')
print(df['baseline_score'].describe().to_string())

# Reason code assignment
def assign_reason_code(row):
    if (row['impressions_90d'] >= 500 and row['avg_position'] > 0 and row['avg_position'] <= 20 and row['ctr'] < 0.5):
        return 'high_impressions_low_ctr'
    elif (row['avg_position'] > 0 and row['avg_position'] <= 10 and row['ctr'] < 0.8 and row['impressions_90d'] >= 300):
        return 'strong_position_weak_ctr'
    elif (row['sessions_90d'] >= 30 and (row['engagement_rate'] < 30 or row['scroll_rate'] < 30) and row['impressions_90d'] >= 100):
        return 'visible_low_engagement'
    elif (row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500):
        return 'stale_visible_page'
    elif (row['avg_position'] > 20 and row['avg_position'] <= 50 and (row['engagement_rate'] >= 50 or row['scroll_rate'] >= 50) and row['sessions_90d'] >= 20):
        return 'position_opportunity_gap'
    else:
        return 'monitor'

def assign_action(reason_code):
    actions = {
        'high_impressions_low_ctr': 'review_ctr',
        'strong_position_weak_ctr': 'review_ctr',
        'visible_low_engagement': 'review_engagement',
        'stale_visible_page': 'refresh',
        'position_opportunity_gap': 'investigate_serp',
        'monitor': 'monitor'
    }
    return actions.get(reason_code, 'monitor')

df['reason_code'] = df.apply(assign_reason_code, axis=1)
df['suggested_action'] = df['reason_code'].apply(assign_action)

reason_counts = df['reason_code'].value_counts().to_dict()
print('Reason code distribution:')
print(json.dumps(reason_counts, indent=2, sort_keys=True))

# Final output columns
output_columns = [
    'content_id',
    'client_id',
    'baseline_rank',
    'baseline_score',
    'reason_code',
    'suggested_action',
    'impressions_90d',
    'avg_position',
    'ctr',
    'engagement_rate',
    'scroll_rate',
    'sessions_90d',
    'days_since_last_update',
    'is_declining_label'
]

final_df = df[output_columns].copy()
final_csv_path = output_dir / 'baseline_action_score.csv'
final_df.to_csv(final_csv_path, index=False)

print(f'✓ Wrote {final_csv_path}')
print(final_df.head(10).to_string(index=False))

# Small metrics JSON as a run receipt
metrics = {
    'rows': int(len(final_df)),
    'top_score': float(final_df['baseline_score'].max()),
    'median_score': float(final_df['baseline_score'].median()),
    'reason_code_counts': reason_counts,
    'top_10': final_df.head(10).to_dict(orient='records')
}
metrics_path = output_dir / 'baseline_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True))
print(f'✓ Wrote {metrics_path}')


## 3. Top-20 review

For each of the top 20 pages in the ranked queue, I write one line describing the action, why it is there, and what would make it wrong. I must be skeptical: the aim is not to make the list look strong, but to check whether the ranking is honest and usable.

Each review should include: action, reason code, confidence note, and one 'what would make it wrong' sentence.


In [ ]:
from pathlib import Path

ranked_path = Path.cwd() / 'work' / 'outputs' / 'baseline_action_score.csv'
if not ranked_path.exists():
    raise FileNotFoundError(f'CSV not found: {ranked_path}')

ranked = pd.read_csv(ranked_path)
top20 = ranked.head(20).copy()

display_cols = [
    'baseline_rank',
    'baseline_score',
    'reason_code',
    'suggested_action',
    'impressions_90d',
    'avg_position',
    'ctr',
    'engagement_rate',
    'days_since_last_update'
]

print(top20[display_cols].to_string(index=False))


## 4. Weak picks + leakage check

The goal here is to read the queue with a skeptic's eye. I look at pages near the cutoff and ask whether they are truly good opportunities or whether the thresholds are too loose. I also confirm that no product flags or future-window variables leaked into the score.

Weak picks are pages that are close to the threshold or provide only one weak signal. If the list is honest, those pages should be lower-confidence and easier to challenge.

Leakage check: no `trend_direction`, `trend_pct`, `health_score`, `needs_ctr_fix`, or future-window features are used. I only use observable inputs available at prediction time.


In [ ]:
ranked = pd.read_csv(Path.cwd() / 'work' / 'outputs' / 'baseline_action_score.csv')

weak = ranked.iloc[14:25].copy()
print('Ranks 15 to 25 (weak pick check):')
print(weak[['baseline_rank', 'baseline_score', 'reason_code', 'suggested_action', 'impressions_90d', 'avg_position', 'ctr', 'engagement_rate']].to_string(index=False))

# Short leakage scan
risk_columns = [c for c in ranked.columns if any(x in c.lower() for x in ['trend', 'health', 'future', 'declin', 'flag'])]
print(f'
Leakage-risk column names: {risk_columns}')
if not risk_columns:
    print('No obvious leakage-risk columns were used in the final output.')

print('
Feature list used by the score:')
print(['impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'sessions_90d', 'days_since_last_update'])


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] I have written the ranked queue to `work/outputs/baseline_action_score.csv`
- [ ] I have not used product flags or future-window variables as features
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
